In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# CELL 1: CLINICAL DEMO SETUP & MODEL LOADING (CORRECTED PREPROCESSING)
import os
import cv2
import numpy as np
import io
import base64
from PIL import Image
from google.colab import files
from IPython.display import display, HTML, clear_output
from tensorflow.keras.models import load_model

print("1. Initializing Clinical Demo Environment...")

# --- FIXED EXACT FILE PATHS ---
eff_path = '/content/drive/MyDrive/best_efficientnet_B3_nih.keras'
cx_path = '/content/drive/MyDrive/best_convnext_nih.keras'

print("2. Waking up EfficientNet-B3... (This takes a few seconds)")
model_eff = load_model(eff_path)
print("✅ EfficientNet-B3 is loaded and ready!")

print("\n3. Waking up ConvNeXt... (This takes a few seconds)")
model_cx = load_model(cx_path)
print("✅ ConvNeXt is loaded and ready!")

def generate_dashboard(img_bytes, filename):
    # Load image and convert to RGB
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Save a copy for the HTML display
    buffered = io.BytesIO()
    img.resize((300, 300)).save(buffered, format="JPEG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode()

    # CRITICAL FIX: Convert to float32, but DO NOT divide by 255!
    # This perfectly mimics how your ImageDataGenerator fed the models.
    img_np = np.array(img, dtype=np.float32)

    # Preprocess for EfficientNet (300x300)
    eff_img = cv2.resize(img_np, (300, 300))
    eff_img = np.expand_dims(eff_img, axis=0)

    # Preprocess for ConvNeXt (224x224)
    cx_img = cv2.resize(img_np, (224, 224))
    cx_img = np.expand_dims(cx_img, axis=0)

    # Get Predictions
    p_eff = model_eff.predict(eff_img, verbose=0)[0][0] * 100
    p_cx = model_cx.predict(cx_img, verbose=0)[0][0] * 100
    p_ens = (0.5 * p_eff) + (0.5 * p_cx)
    variance = abs(p_eff - p_cx)

    # Clinical Logic
    if variance > 30.0:
        status_color = "#f39c12"
        diagnosis = "⚠️ BORDERLINE / SUSPICIOUS"
        action = "High variance between AI Specialists. Manual radiologist review strictly required."
    elif p_ens >= 60.0:
        status_color = "#e74c3c"
        diagnosis = "🚨 EFFUSION DETECTED"
        action = "Immediate clinical review and patient follow-up recommended."
    elif p_ens <= 40.0:
        status_color = "#2ecc71"
        diagnosis = "✅ NORMAL (NO FINDINGS)"
        action = "Standard routine follow-up. No immediate action required."
    else:
        status_color = "#f1c40f"
        diagnosis = "⚠️ UNCERTAIN (LOW CONFIDENCE)"
        action = "Ensemble confidence is weak. Requesting secondary scan or manual review."

    # HTML UI Generation
    html_out = f"""
    <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; max-width: 800px; margin: 20px auto; border: 1px solid #ddd; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); overflow: hidden; background-color: #fff;">
        <div style="background-color: #2c3e50; color: white; padding: 15px 20px; text-align: center;">
            <h2 style="margin: 0; font-size: 22px; letter-spacing: 1px;">AUTOMATED MEDICAL DIAGNOSTICS REPORT</h2>
            <p style="margin: 5px 0 0 0; font-size: 12px; opacity: 0.8;">File: {filename}</p>
        </div>

        <div style="display: flex; padding: 20px; gap: 20px;">
            <div style="flex: 1; text-align: center;">
                <img src="data:image/jpeg;base64,{img_b64}" style="width: 100%; max-width: 300px; border-radius: 5px; border: 1px solid #ccc;"/>
                <p style="margin-top: 10px; font-weight: bold; color: #555;">Patient X-Ray Scan</p>
            </div>

            <div style="flex: 1; display: flex; flex-direction: column; justify-content: center;">
                <h3 style="margin-top: 0; color: #333; border-bottom: 2px solid #eee; padding-bottom: 5px;">AI Specialist Confidence</h3>

                <div style="margin-bottom: 10px;">
                    <span style="font-weight: bold; font-size: 14px;">Doctor 1 (EfficientNet):</span>
                    <span style="float: right; font-weight: bold;">{p_eff:.2f}%</span>
                    <div style="background-color: #ecf0f1; border-radius: 4px; height: 10px; width: 100%; margin-top: 5px;">
                        <div style="background-color: #3498db; width: {p_eff}%; height: 100%; border-radius: 4px;"></div>
                    </div>
                </div>

                <div style="margin-bottom: 15px;">
                    <span style="font-weight: bold; font-size: 14px;">Doctor 2 (ConvNeXt):</span>
                    <span style="float: right; font-weight: bold;">{p_cx:.2f}%</span>
                    <div style="background-color: #ecf0f1; border-radius: 4px; height: 10px; width: 100%; margin-top: 5px;">
                        <div style="background-color: #3498db; width: {p_cx}%; height: 100%; border-radius: 4px;"></div>
                    </div>
                </div>

                <div style="background-color: #f8f9fa; padding: 10px; border-left: 4px solid #34495e; border-radius: 4px; margin-bottom: 20px;">
                    <span style="font-weight: bold; font-size: 16px;">Ensemble Average:</span>
                    <span style="float: right; font-weight: bold; font-size: 16px;">{p_ens:.2f}%</span>
                </div>

                <div style="background-color: {status_color}20; border: 2px solid {status_color}; border-radius: 6px; padding: 15px;">
                    <h3 style="color: {status_color}; margin: 0 0 10px 0; font-size: 18px;">{diagnosis}</h3>
                    <p style="margin: 0; font-size: 14px; color: #444;"><strong>Action Required:</strong> {action}</p>
                </div>
            </div>
        </div>
    </div>
    """
    display(HTML(html_out))

print("\n✅ System Online. Proceed to run the Interactive UI Cell.")

1. Initializing Clinical Demo Environment...
2. Waking up EfficientNet-B3... (This takes a few seconds)
✅ EfficientNet-B3 is loaded and ready!

3. Waking up ConvNeXt... (This takes a few seconds)
✅ ConvNeXt is loaded and ready!

✅ System Online. Proceed to run the Interactive UI Cell.


In [8]:
# CELL 2: THE LIVE INTERACTIVE APP UI
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create a sleek, restricted Image Uploader
uploader = widgets.FileUpload(
    accept='image/*',  # Strictly restricts to images (.png, .jpg, .jpeg)
    multiple=False,    # STRICTLY forces ONE image at a time
    description='Upload X-Ray',
    style={'button_color': '#2ecc71', 'font_weight': 'bold'},
    layout={'width': '250px'}
)

# 2. Create an output area where the dashboard will render
output_area = widgets.Output()

# 3. Define what happens when an image is uploaded
def on_upload_change(change):
    with output_area:
        clear_output(wait=True) # Clears the previous patient's dashboard

        # Check if an image was actually uploaded
        if uploader.value:
            # Handle Colab's widget versioning safely
            uploaded_data = uploader.value
            if isinstance(uploaded_data, tuple): # Newer ipywidgets
                img_bytes = uploaded_data[0]['content']
                filename = uploaded_data[0]['name']
            else: # Older ipywidgets
                filename = list(uploaded_data.keys())[0]
                img_bytes = uploaded_data[filename]['content']

            print(f"⏳ Processing patient scan: {filename}...")

            # Generate the beautiful HTML dashboard from Cell 1
            generate_dashboard(img_bytes, filename)

            # Clear the uploader's memory so it's ready for the next single image
            uploader.value.clear()

# 4. Attach the function to the button
uploader.observe(on_upload_change, names='value')

# 5. Display the final App UI
display(widgets.HTML("""
    <div style="margin-bottom: 10px;">
        <h3 style="color: #2c3e50; margin-bottom: 5px;">🏥 Live Clinical Inference Engine</h3>
        <p style="color: #7f8c8d; font-size: 14px; margin-top: 0;">Please select a <b>single</b> X-Ray image for AI evaluation.</p>
    </div>
"""))
display(uploader)
display(output_area)

HTML(value='\n    <div style="margin-bottom: 10px;">\n        <h3 style="color: #2c3e50; margin-bottom: 5px;">…

FileUpload(value={}, accept='image/*', description='Upload X-Ray', layout=Layout(width='250px'), style=ButtonS…

Output()